In [2]:
!pip install opencv-python

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 24.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 42.7 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scipy 1.10.1 requires numpy<1.27.0,>=1.19.5, but you have numpy 2.2.6 which is incompatible.
pyfume 0.3.4 requires numpy==1.24.4, but you have numpy 2.2.6 which is incompatible.
numba 0.56.4 requires numpy<1.24,>=1.18, but you have numpy 2.2.6 which is incompatible.


## Baseline Model

In [10]:
import cv2
import os
import time
import random
import math

# ============================================================================
# BASELINE CNN (RANDOM WEIGHTS)
# ============================================================================

def load_img(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (32, 32))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.normalize(img.astype('float32'), None, 0.0, 1.0, cv2.NORM_MINMAX)
    return [[list(map(float, img[i, :, c])) for i in range(32)] for c in range(3)]

class BaselineConv2D:
    def __init__(self, in_c, out_c, k=3, p=1, h=16, w=16):
        self.in_c, self.out_c, self.k, self.p = in_c, out_c, k, p
        self.h, self.w = h, w
        # Random weights - NEVER UPDATED!
        s = (2.0 / (in_c * k * k)) ** 0.5
        self.weights = [[[[random.gauss(0, s) for _ in range(k)] for _ in range(k)]
                   for _ in range(in_c)] for _ in range(out_c)]
        self.b = [0.0] * out_c

    def forward(self, x):
        batch_size = len(x)
        height = self.h
        width = self.w
        
        out = []
        for _ in range(batch_size):
            batch_out = []
            for oc in range(self.out_c):
                channel_out = []
                for _ in range(height):
                    row = [self.b[oc]] * width
                    channel_out.append(row)
                batch_out.append(channel_out)
            out.append(batch_out)

        for bi in range(batch_size):
            x_batch = x[bi]
            out_batch = out[bi]
            for oc in range(self.out_c):
                w_oc = self.weights[oc]
                out_oc = out_batch[oc]
                for ic in range(self.in_c):
                    w_ic = w_oc[ic]
                    x_ic = x_batch[ic]
                    for i in range(height):
                        out_row = out_oc[i]
                        for j in range(width):
                            s = 0.0
                            for ki in range(self.k):
                                ii = i + ki - self.p
                                if 0 <= ii < height:
                                    w_row = w_ic[ki]
                                    x_row = x_ic[ii]
                                    for kj in range(self.k):
                                        jj = j + kj - self.p
                                        if 0 <= jj < width:
                                            s += x_row[jj] * w_row[kj]
                            out_row[j] += s
        return out


class BaselinePool2D:
    def __init__(self, k=2):
        self.k = k

    def forward(self, x):
        b, c = len(x), len(x[0])
        h, w = len(x[0][0]), len(x[0][0][0])
        oh, ow = h // self.k, w // self.k
        
        out = [[[[0.0]*ow for _ in range(oh)] for _ in range(c)] for _ in range(b)]
        
        for bi in range(b):
            x_batch = x[bi]
            out_batch = out[bi]
            
            for ci in range(c):
                x_c = x_batch[ci]
                out_c = out_batch[ci]
                
                for i in range(oh):
                    ii = i * self.k
                    out_row = out_c[i]
                    
                    for j in range(ow):
                        jj = j * self.k
                        max_val = x_c[ii][jj]
                        
                        for ki in range(self.k):
                            x_row = x_c[ii + ki]
                            for kj in range(self.k):
                                val = x_row[jj + kj]
                                if val > max_val:
                                    max_val = val
                        
                        out_row[j] = max_val
        return out


class BaselineDense:
    def __init__(self, in_f, out_f):
        self.in_f, self.out_f = in_f, out_f
        # Random weights - NEVER UPDATED!
        s = (2.0 / in_f) ** 0.5
        self.w = [[random.gauss(0, s) for _ in range(in_f)] for _ in range(out_f)]
        self.b = [0.0] * out_f

    def forward(self, x):
        b = len(x)
        out = [[self.b[j] for j in range(self.out_f)] for _ in range(b)]
        
        for bi in range(b):
            x_b = x[bi]
            out_b = out[bi]
            
            for j in range(self.out_f):
                w_j = self.w[j]
                s = 0.0
                for i in range(self.in_f):
                    s += x_b[i] * w_j[i]
                out_b[j] += s
        return out


def relu_fwd(x):
    return [[[[max(0.0, x[b][c][i][j])
               for j in range(len(x[0][0][0]))]
              for i in range(len(x[0][0]))]
             for c in range(len(x[0]))]
            for b in range(len(x))]

def flatten_fwd(x):
    return [[x[b][c][i][j]
             for c in range(len(x[0]))
             for i in range(len(x[0][0]))
             for j in range(len(x[0][0][0]))]
            for b in range(len(x))]

def softmax_fwd(x):
    out = []
    for b in x:
        mx = max(b)
        ex = [math.exp(v - mx) for v in b]
        s = sum(ex)
        inv_s = 1.0 / s
        out.append([e * inv_s for e in ex])
    return out

def prep_files(path, ratio=0.8):
    fps, lbls = [], []
    classes = sorted([d for d in os.listdir(path)
                      if os.path.isdir(os.path.join(path, d))])

    if not classes:
        print(f"ERROR: No class directories found in {path}")
        return [], [], [], [], 0

    for idx, cls in enumerate(classes):
        cpath = os.path.join(path, cls)
        for name in os.listdir(cpath):
            if not name.endswith(('.png','.jpg','.jpeg')): continue
            fps.append(os.path.join(cpath, name))
            lbls.append(idx)

    if not fps:
        print(f"ERROR: No images found in {path}")
        return [], [], [], [], 0

    combined = list(zip(fps, lbls))
    random.shuffle(combined)
    fps, lbls = zip(*combined)
    fps, lbls = list(fps), list(lbls)

    split = int(len(fps) * ratio)
    print(f"Train: {split}, Test: {len(fps)-split}, Classes: {len(classes)}")
    return fps[:split], lbls[:split], fps[split:], lbls[split:], len(classes)

def load_batch(paths, labels, indices):
    imgs, lbls = [], []
    for i in indices:
        img = load_img(paths[i])
        if img is not None:
            imgs.append(img)
            lbls.append(labels[i])
    return imgs, lbls

def baseline_test(tr_p, tr_l, n_cls, bs=256):
    """Test with random weights - NO TRAINING!"""
    num_filters = 4
    
    # Initialize with random weights
    pool0 = BaselinePool2D(2)
    conv = BaselineConv2D(3, num_filters, 3, 1, h=16, w=16)
    pool1 = BaselinePool2D(2)
    dense_input = num_filters * 8 * 8
    dense = BaselineDense(dense_input, n_cls)

    # Calculate complexity
    conv_macs = 3 * num_filters * 9 * 16 * 16
    dense_macs = dense_input * n_cls
    total_macs = conv_macs + dense_macs
    params = (3 * num_filters * 9 + num_filters) + (dense_input * n_cls + n_cls)
    
    print(f"\nBASELINE CNN (RANDOM WEIGHTS):")
    print(f"  Architecture: Pool(32→16) → Conv(16×16, 3→{num_filters}) → Pool(16→8) → Dense({dense_input}→{n_cls})")
    print(f"\nCOMPLEXITY:")
    print(f"  TOTAL Params: {params:,}")
    print(f"  TOTAL MACs:   {total_macs:,}")
    print(f"  TOTAL FLOPs:  {total_macs*2:,}")

    t0 = time.time()
    corr = tot = 0

    # Just one pass through data - NO TRAINING!
    for i in range(0, len(tr_p), bs):
        bi = list(range(i, min(i+bs, len(tr_p))))
        bx, blbl = load_batch(tr_p, tr_l, bi)
        if not bx: continue

        # Forward only
        p0 = pool0.forward(bx)
        c1 = conv.forward(p0)
        c1r = relu_fwd(c1)
        p1 = pool1.forward(c1r)
        fl = flatten_fwd(p1)
        out = dense.forward(fl)
        preds = softmax_fwd(out)

        # Accuracy
        for j, p in enumerate(preds):
            if p.index(max(p)) == blbl[j]: corr += 1
        tot += len(bx)

    acc = 100.0 * corr / tot
    elapsed = time.time() - t0
    
    print(f"Baseline Accuracy (Random Weights): {acc:.2f}%")
    print(f"Time: {elapsed/60:.1f} min")
    
    return pool0, conv, pool1, dense, acc

def evaluate(te_p, te_l, pool0, conv, pool1, dense, bs=256):
    corr = tot = 0
    for i in range(0, len(te_p), bs):
        bi = list(range(i, min(i+bs, len(te_p))))
        bx, blbl = load_batch(te_p, te_l, bi)
        if not bx: continue

        p0 = pool0.forward(bx)
        c1 = conv.forward(p0)
        c1r = relu_fwd(c1)
        p1 = pool1.forward(c1r)
        fl = flatten_fwd(p1)
        out = dense.forward(fl)
        preds = softmax_fwd(out)

        for j, p in enumerate(preds):
            if p.index(max(p)) == blbl[j]: corr += 1
        tot += len(bx)

    print(f"Test Accuracy (Random Weights): {100.0*corr/tot:.2f}%")

# ============================================================================
# MAIN
# ============================================================================

path = '/scratch/karthikl_NEW/aditig/ERA5_weekly/data_1/'

print("="*70)
print("BASELINE CNN (RANDOM WEIGHTS ASSIGNMENT)")
print("="*70)

tr_p, tr_l, te_p, te_l, n_cls = prep_files(path)

print(f"\nRunning baseline model")
pool0, conv, pool1, dense, train_acc = baseline_test(tr_p, tr_l, n_cls, bs=256)

print(f"\n{'='*70}")
print("BASELINE RESULTS:")
print("="*70)

evaluate(te_p, te_l, pool0, conv, pool1, dense)

print(f"\n{'='*70}")
print("COMPARISON:")
print(f"  This Baseline:     {train_acc:.2f}%")
print("="*70)

BASELINE CNN - NO BACKPROP (RANDOM WEIGHTS ONLY)
This baseline shows performance WITHOUT training.
Compare this to your trained model to see the improvement!
Train: 48000, Test: 12000, Classes: 10

Running baseline (forward pass only, no training)...

BASELINE CNN (NO BACKPROP - RANDOM WEIGHTS):
  Architecture: Pool(32→16) → Conv(16×16, 3→4) → Pool(16→8) → Dense(256→10)

COMPLEXITY:
  TOTAL Params: 2,682
  TOTAL MACs:   30,208
  TOTAL FLOPs:  60,416

⚠️  NO TRAINING - JUST FORWARD PASS WITH RANDOM WEIGHTS!

Baseline Accuracy (Random Weights): 15.46%
Time: 8.7 min
Expected Random Accuracy: 10.00%

BASELINE RESULTS:
Test Accuracy (Random Weights): 15.28%

COMPARISON:
  Random Baseline:  ~10.00% (expected)
  This Baseline:     15.46%
  Your Trained Model: ~18-22% (with backprop)
  Improvement:       ~4.54% gain from training!


In [10]:
import cv2
import os
import time
import random
import math

def load_img(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (32, 32))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.normalize(img.astype('float32'), None, 0.0, 1.0, cv2.NORM_MINMAX)
    return [[list(map(float, img[i, :, c])) for i in range(32)] for c in range(3)]

class TinyConv2D:
    def __init__(self, in_c, out_c, k=3, p=1, h=16, w=16):
        self.in_c, self.out_c, self.k, self.p = in_c, out_c, k, p
        self.h, self.w = h, w
        s = (2.0 / (in_c * k * k)) ** 0.5
        self.weights = [[[[random.gauss(0, s) for _ in range(k)] for _ in range(k)]
                   for _ in range(in_c)] for _ in range(out_c)]
        self.b = [0.0] * out_c
        self.x = None

    def forward(self, x):
        self.x = x
        batch_size = len(x)
        height = self.h
        width = self.w
        
        out = []
        for _ in range(batch_size):
            batch_out = []
            for oc in range(self.out_c):
                channel_out = []
                for _ in range(height):
                    row = [self.b[oc]] * width
                    channel_out.append(row)
                batch_out.append(channel_out)
            out.append(batch_out)

        for bi in range(batch_size):
            x_batch = x[bi]
            out_batch = out[bi]
            for oc in range(self.out_c):
                w_oc = self.weights[oc]
                out_oc = out_batch[oc]
                for ic in range(self.in_c):
                    w_ic = w_oc[ic]
                    x_ic = x_batch[ic]
                    for i in range(height):
                        out_row = out_oc[i]
                        for j in range(width):
                            s = 0.0
                            for ki in range(self.k):
                                ii = i + ki - self.p
                                if 0 <= ii < height:
                                    w_row = w_ic[ki]
                                    x_row = x_ic[ii]
                                    for kj in range(self.k):
                                        jj = j + kj - self.p
                                        if 0 <= jj < width:
                                            s += x_row[jj] * w_row[kj]
                            out_row[j] += s
        return out

    def backward(self, dout, lr):
        batch_size = len(dout)
        height = self.h
        width = self.w
        norm = lr / (batch_size * height * width)
        
        d_w = [[[[0.0 for _ in range(self.k)] for _ in range(self.k)]
                for _ in range(self.in_c)] for _ in range(self.out_c)]
        d_b = [0.0] * self.out_c
        
        for oc in range(self.out_c):
            gb = 0.0
            d_w_oc = d_w[oc]
            
            for bi in range(batch_size):
                x_batch = self.x[bi]
                dout_batch = dout[bi]
                dout_oc = dout_batch[oc]
                
                for i in range(height):
                    dout_row = dout_oc[i]
                    for j in range(width):
                        grad = dout_row[j]
                        if grad > 5.0:
                            grad = 5.0
                        elif grad < -5.0:
                            grad = -5.0
                        
                        gb += grad
                        
                        for ic in range(self.in_c):
                            x_ic = x_batch[ic]
                            d_w_ic = d_w_oc[ic]
                            
                            for ki in range(self.k):
                                ii = i + ki - self.p
                                if 0 <= ii < height:
                                    x_row = x_ic[ii]
                                    d_w_row = d_w_ic[ki]
                                    for kj in range(self.k):
                                        jj = j + kj - self.p
                                        if 0 <= jj < width:
                                            d_w_row[kj] += grad * x_row[jj]
            
            self.b[oc] -= norm * gb
            
            for ic in range(self.in_c):
                w_ic = self.weights[oc][ic]
                d_w_ic = d_w_oc[ic]
                for ki in range(self.k):
                    w_row = w_ic[ki]
                    d_w_row = d_w_ic[ki]
                    for kj in range(self.k):
                        w_row[kj] -= norm * d_w_row[kj]


class TinyPool2D:
    def __init__(self, k=2):
        self.k = k
        self.max_idx = None

    def forward(self, x):
        b, c = len(x), len(x[0])
        h, w = len(x[0][0]), len(x[0][0][0])
        oh, ow = h // self.k, w // self.k
        
        out = [[[[0.0]*ow for _ in range(oh)] for _ in range(c)] for _ in range(b)]
        self.max_idx = [[[[None]*ow for _ in range(oh)] for _ in range(c)] for _ in range(b)]
        
        for bi in range(b):
            x_batch = x[bi]
            out_batch = out[bi]
            idx_batch = self.max_idx[bi]
            
            for ci in range(c):
                x_c = x_batch[ci]
                out_c = out_batch[ci]
                idx_c = idx_batch[ci]
                
                for i in range(oh):
                    ii = i * self.k
                    out_row = out_c[i]
                    idx_row = idx_c[i]
                    
                    for j in range(ow):
                        jj = j * self.k
                        max_val = x_c[ii][jj]
                        max_pos = (ii, jj)
                        
                        for ki in range(self.k):
                            x_row = x_c[ii + ki]
                            for kj in range(self.k):
                                val = x_row[jj + kj]
                                if val > max_val:
                                    max_val = val
                                    max_pos = (ii + ki, jj + kj)
                        
                        out_row[j] = max_val
                        idx_row[j] = max_pos
        return out

    def backward(self, dout):
        b, c = len(dout), len(dout[0])
        oh, ow = len(dout[0][0]), len(dout[0][0][0])
        h, w = oh * self.k, ow * self.k
        
        dx = [[[[0.0]*w for _ in range(h)] for _ in range(c)] for _ in range(b)]
        
        for bi in range(b):
            dout_batch = dout[bi]
            dx_batch = dx[bi]
            idx_batch = self.max_idx[bi]
            
            for ci in range(c):
                dout_c = dout_batch[ci]
                dx_c = dx_batch[ci]
                idx_c = idx_batch[ci]
                
                for i in range(oh):
                    dout_row = dout_c[i]
                    idx_row = idx_c[i]
                    
                    for j in range(ow):
                        ii, jj = idx_row[j]
                        dx_c[ii][jj] = dout_row[j]
        return dx


class TinyDense:
    def __init__(self, in_f, out_f):
        self.in_f, self.out_f = in_f, out_f
        s = (2.0 / in_f) ** 0.5
        self.w = [[random.gauss(0, s) for _ in range(in_f)] for _ in range(out_f)]
        self.b = [0.0] * out_f
        self.x = None

    def forward(self, x):
        self.x = x
        b = len(x)
        out = [[self.b[j] for j in range(self.out_f)] for _ in range(b)]
        
        for bi in range(b):
            x_b = x[bi]
            out_b = out[bi]
            
            for j in range(self.out_f):
                w_j = self.w[j]
                s = 0.0
                for i in range(self.in_f):
                    s += x_b[i] * w_j[i]
                out_b[j] += s
        return out

    def backward(self, dout, lr):
        b = len(dout)
        norm = lr / b
        
        d_w = [[0.0 for _ in range(self.in_f)] for _ in range(self.out_f)]
        d_b = [0.0] * self.out_f
        dx = [[0.0 for _ in range(self.in_f)] for _ in range(b)]
        
        for j in range(self.out_f):
            gb = 0.0
            w_j = self.w[j]
            d_w_j = d_w[j]
            
            for bi in range(b):
                grad = dout[bi][j]
                if grad > 5.0:
                    grad = 5.0
                elif grad < -5.0:
                    grad = -5.0
                
                gb += grad
                x_b = self.x[bi]
                dx_b = dx[bi]
                
                for i in range(self.in_f):
                    d_w_j[i] += grad * x_b[i]
                    dx_b[i] += grad * w_j[i]
            
            self.b[j] -= norm * gb
            
            for i in range(self.in_f):
                w_j[i] -= norm * d_w_j[i]
        
        return dx


def relu_fwd(x):
    return [[[[max(0.0, x[b][c][i][j])
               for j in range(len(x[0][0][0]))]
              for i in range(len(x[0][0]))]
             for c in range(len(x[0]))]
            for b in range(len(x))]

def relu_bwd(dout, x_pre):
    return [[[[dout[b][c][i][j] if x_pre[b][c][i][j] > 0 else 0.0
               for j in range(len(dout[0][0][0]))]
              for i in range(len(dout[0][0]))]
             for c in range(len(dout[0]))]
            for b in range(len(dout))]

def flatten_fwd(x):
    return [[x[b][c][i][j]
             for c in range(len(x[0]))
             for i in range(len(x[0][0]))
             for j in range(len(x[0][0][0]))]
            for b in range(len(x))]

def unflatten(dx, c, h, w):
    out = [[[[0.0]*w for _ in range(h)] for _ in range(c)] for _ in range(len(dx))]
    for b in range(len(dx)):
        dx_b = dx[b]
        out_b = out[b]
        idx = 0
        for ci in range(c):
            out_c = out_b[ci]
            for i in range(h):
                out_row = out_c[i]
                for j in range(w):
                    out_row[j] = dx_b[idx]
                    idx += 1
    return out

def ce_loss_grad(preds, tgts):
    b = len(preds)
    loss = 0.0
    grads = [[0.0]*len(preds[0]) for _ in range(b)]
    
    for i in range(b):
        pred_i = preds[i]
        mx = max(pred_i)
        ex = [math.exp(p - mx) for p in pred_i]
        s = sum(ex)
        inv_s = 1.0 / s
        probs = [e * inv_s for e in ex]
        
        loss -= math.log(probs[tgts[i]] + 1e-8)
        
        grad_i = grads[i]
        for j in range(len(pred_i)):
            grad_i[j] = probs[j]
        grad_i[tgts[i]] -= 1.0
    
    return loss / b, grads

def softmax_fwd(x):
    out = []
    for b in x:
        mx = max(b)
        ex = [math.exp(v - mx) for v in b]
        s = sum(ex)
        inv_s = 1.0 / s
        out.append([e * inv_s for e in ex])
    return out

def prep_files(path, ratio=0.8):
    fps, lbls = [], []
    classes = sorted([d for d in os.listdir(path)
                      if os.path.isdir(os.path.join(path, d))])

    for idx, cls in enumerate(classes):
        cpath = os.path.join(path, cls)
        for name in os.listdir(cpath):
            if not name.endswith(('.png','.jpg','.jpeg')): continue
            fps.append(os.path.join(cpath, name))
            lbls.append(idx)

    combined = list(zip(fps, lbls))
    random.shuffle(combined)
    fps, lbls = zip(*combined)
    fps, lbls = list(fps), list(lbls)

    split = int(len(fps) * ratio)
    print(f"Train: {split}, Test: {len(fps)-split}, Classes: {len(classes)}")
    return fps[:split], lbls[:split], fps[split:], lbls[split:], len(classes)

def load_batch(paths, labels, indices):
    imgs, lbls = [], []
    for i in indices:
        img = load_img(paths[i])
        if img is not None:
            imgs.append(img)
            lbls.append(labels[i])
    return imgs, lbls

def train(tr_p, tr_l, n_cls, epochs=2, bs=256, lr=0.08):
    # Pool BEFORE conv!
    num_filters = 4
    
    pool0 = TinyPool2D(2)  # 32x32 -> 16x16 FIRST (0 MACs!)
    conv = TinyConv2D(3, num_filters, 3, 1, h=16, w=16)  # Conv on 16x16 (4x fewer MACs!)
    pool1 = TinyPool2D(2)  # 16x16 -> 8x8
    dense_input = num_filters * 8 * 8  # 256 features
    dense = TinyDense(dense_input, n_cls)

    # Calculate MACs
    conv_macs = 3 * num_filters * 9 * 16 * 16  # On 16x16!
    dense_macs = dense_input * n_cls
    total_macs = conv_macs + dense_macs
    params = (3 * num_filters * 9 + num_filters) + (dense_input * n_cls + n_cls)
    
    print(f"\nULTRA-LOW MAC/FLOP CNN:")
    print(f"  Architecture: Pool(32→16) → Conv(16×16, 3→{num_filters}) → Pool(16→8) → Dense({dense_input}→{n_cls})")
    print(f"\nCOMPLEXITY:")
    print(f"  Conv MACs:  {conv_macs:,} (on 16×16)")
    print(f"  Dense MACs: {dense_macs:,}")
    print(f"  ─────────────────────────")
    print(f"  TOTAL Params: {params:,}")
    print(f"  TOTAL MACs:   {total_macs:,}")
    print(f"  TOTAL FLOPs:  {total_macs*2:,}")
    print(f"\nTRAINING:")
    print(f"  Epochs: {epochs}, Batch: {bs}, LR: {lr}")

    for ep in range(epochs):
        t0 = time.time()
        loss_sum = corr = tot = 0

        idx = list(range(len(tr_p)))
        random.shuffle(idx)
        
        num_batches = len(tr_p) // bs

        for i in range(0, len(tr_p), bs):
            bi = idx[i:i+bs]
            bx, blbl = load_batch(tr_p, tr_l, bi)
            if not bx: continue
            
            batch_num = i // bs
            if batch_num % 30 == 0:
                elapsed = time.time() - t0
                rate = batch_num / elapsed if elapsed > 0 else 0
                print(f"  Batch {batch_num}/{num_batches} | Elapsed: {elapsed/60:.1f}min)

            # Forward
            p0 = pool0.forward(bx)  # Pool FIRST
            c1 = conv.forward(p0)   # Conv on 16x16
            c1r = relu_fwd(c1)
            p1 = pool1.forward(c1r)
            fl = flatten_fwd(p1)
            out = dense.forward(fl)

            # Loss
            loss, dout = ce_loss_grad(out, blbl)
            loss_sum += loss * len(bx)
            tot += len(bx)

            # Backward
            dx_flat = dense.backward(dout, lr)
            dx_unflat = unflatten(dx_flat, num_filters, 8, 8)
            dx_pool1 = pool1.backward(dx_unflat)
            dx_relu = relu_bwd(dx_pool1, c1)
            conv.backward(dx_relu, lr)

            # Accuracy
            preds = softmax_fwd(out)
            for j, p in enumerate(preds):
                if p.index(max(p)) == blbl[j]: corr += 1

        print(f"\nEpoch {ep+1}/{epochs} | Loss: {loss_sum/tot:.4f} | "
              f"Acc: {100.0*corr/tot:.2f}% | Time: {(time.time()-t0)/60:.1f}min")

    return pool0, conv, pool1, dense

def evaluate(te_p, te_l, pool0, conv, pool1, dense, bs=256):
    corr = tot = 0
    for i in range(0, len(te_p), bs):
        bi = list(range(i, min(i+bs, len(te_p))))
        bx, blbl = load_batch(te_p, te_l, bi)
        if not bx: continue

        p0 = pool0.forward(bx)
        c1 = conv.forward(p0)
        c1r = relu_fwd(c1)
        p1 = pool1.forward(c1r)
        fl = flatten_fwd(p1)
        out = dense.forward(fl)
        preds = softmax_fwd(out)

        for j, p in enumerate(preds):
            if p.index(max(p)) == blbl[j]: corr += 1
        tot += len(bx)

    print(f"\nTest Acc: {100.0*corr/tot:.2f}%")

# ============================================================================
# MAIN
# ============================================================================

path = '/scratch/karthikl_NEW/aditig/ERA5_weekly/data_1/'

print("="*70)
print("ULTRA-LOW MAC/FLOP CNN")
print("="*70)

tr_p, tr_l, te_p, te_l, n_cls = prep_files(path)

t0 = time.time()
pool0, conv, pool1, dense = train(tr_p, tr_l, n_cls, epochs=2, bs=256, lr=0.08)
total_time = (time.time()-t0)/60

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE: {total_time:.1f} min ({total_time/60:.2f} hours)")
print("="*70)

evaluate(te_p, te_l, pool0, conv, pool1, dense)

if total_time < 180:
    print(f"\n✓ SUCCESS: Completed in under 3 hours!")
    print(f"  Time used: {total_time:.1f} min")
    print(f"  Margin: {180-total_time:.1f} min remaining")
else:
    print(f"\n⚠ Exceeded 3 hour limit: {total_time:.1f} min")

ULTRA-LOW MAC/FLOP CNN
Train: 48000, Test: 12000, Classes: 10

ULTRA-LOW MAC/FLOP CNN:
  Architecture: Pool(32→16) → Conv(16×16, 3→4) → Pool(16→8) → Dense(256→10)

COMPLEXITY:
  Conv MACs:  27,648 (on 16×16, not 32×32!)
  Dense MACs: 2,560
  ─────────────────────────
  TOTAL Params: 2,682
  TOTAL MACs:   30,208
  TOTAL FLOPs:  60,416

TRAINING:
  Epochs: 2, Batch: 256, LR: 0.08
  Expected: ~25-30 min/epoch

  Batch 180/187 | Elapsed: 16.7min | ETA: 0.7min
Epoch 1/2 | Loss: 1.5724 | Acc: 55.53% | Time: 17.4min
  Batch 180/187 | Elapsed: 11.1min | ETA: 0.4min
Epoch 2/2 | Loss: 1.0536 | Acc: 70.76% | Time: 11.6min

TRAINING COMPLETE: 29.0 min (0.48 hours)

Test Acc: 72.88%

✓ SUCCESS: Completed in under 3 hours!
  Time used: 29.0 min
  Margin: 151.0 min remaining


In [2]:
import cv2
import os
import time
import random
import math


def load_img(path):
    img = cv2.imread(path)
    if img is None:
        return None
    img = cv2.resize(img, (32, 32))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.normalize(img.astype('float32'), None, 0.0, 1.0, cv2.NORM_MINMAX)
    return [[list(map(float, img[i, :, c])) for i in range(32)] for c in range(3)]

class TinyConv2D:
    def __init__(self, in_c, out_c, k=3, p=1, h=16, w=16):
        self.in_c, self.out_c, self.k, self.p = in_c, out_c, k, p
        self.h, self.w = h, w
        s = (2.0 / (in_c * k * k)) ** 0.5
        self.weights = [[[[random.gauss(0, s) for _ in range(k)] for _ in range(k)]
                   for _ in range(in_c)] for _ in range(out_c)]
        self.b = [0.0] * out_c
        self.x = None

    def forward(self, x):
        self.x = x
        batch_size = len(x)
        height = self.h
        width = self.w
        
        out = []
        for _ in range(batch_size):
            batch_out = []
            for oc in range(self.out_c):
                channel_out = []
                for _ in range(height):
                    row = [self.b[oc]] * width
                    channel_out.append(row)
                batch_out.append(channel_out)
            out.append(batch_out)

        for bi in range(batch_size):
            x_batch = x[bi]
            out_batch = out[bi]
            for oc in range(self.out_c):
                w_oc = self.weights[oc]
                out_oc = out_batch[oc]
                for ic in range(self.in_c):
                    w_ic = w_oc[ic]
                    x_ic = x_batch[ic]
                    for i in range(height):
                        out_row = out_oc[i]
                        for j in range(width):
                            s = 0.0
                            for ki in range(self.k):
                                ii = i + ki - self.p
                                if 0 <= ii < height:
                                    w_row = w_ic[ki]
                                    x_row = x_ic[ii]
                                    for kj in range(self.k):
                                        jj = j + kj - self.p
                                        if 0 <= jj < width:
                                            s += x_row[jj] * w_row[kj]
                            out_row[j] += s
        return out

    def backward(self, dout, lr):
        batch_size = len(dout)
        height = self.h
        width = self.w
        norm = lr / (batch_size * height * width)
        
        d_w = [[[[0.0 for _ in range(self.k)] for _ in range(self.k)]
                for _ in range(self.in_c)] for _ in range(self.out_c)]
        d_b = [0.0] * self.out_c
        
        for oc in range(self.out_c):
            gb = 0.0
            d_w_oc = d_w[oc]
            
            for bi in range(batch_size):
                x_batch = self.x[bi]
                dout_batch = dout[bi]
                dout_oc = dout_batch[oc]
                
                for i in range(height):
                    dout_row = dout_oc[i]
                    for j in range(width):
                        grad = dout_row[j]
                        if grad > 5.0:
                            grad = 5.0
                        elif grad < -5.0:
                            grad = -5.0
                        
                        gb += grad
                        
                        for ic in range(self.in_c):
                            x_ic = x_batch[ic]
                            d_w_ic = d_w_oc[ic]
                            
                            for ki in range(self.k):
                                ii = i + ki - self.p
                                if 0 <= ii < height:
                                    x_row = x_ic[ii]
                                    d_w_row = d_w_ic[ki]
                                    for kj in range(self.k):
                                        jj = j + kj - self.p
                                        if 0 <= jj < width:
                                            d_w_row[kj] += grad * x_row[jj]
            
            self.b[oc] -= norm * gb
            
            for ic in range(self.in_c):
                w_ic = self.weights[oc][ic]
                d_w_ic = d_w_oc[ic]
                for ki in range(self.k):
                    w_row = w_ic[ki]
                    d_w_row = d_w_ic[ki]
                    for kj in range(self.k):
                        w_row[kj] -= norm * d_w_row[kj]


class TinyPool2D:
    def __init__(self, k=2):
        self.k = k
        self.max_idx = None

    def forward(self, x):
        b, c = len(x), len(x[0])
        h, w = len(x[0][0]), len(x[0][0][0])
        oh, ow = h // self.k, w // self.k
        
        out = [[[[0.0]*ow for _ in range(oh)] for _ in range(c)] for _ in range(b)]
        self.max_idx = [[[[None]*ow for _ in range(oh)] for _ in range(c)] for _ in range(b)]
        
        for bi in range(b):
            x_batch = x[bi]
            out_batch = out[bi]
            idx_batch = self.max_idx[bi]
            
            for ci in range(c):
                x_c = x_batch[ci]
                out_c = out_batch[ci]
                idx_c = idx_batch[ci]
                
                for i in range(oh):
                    ii = i * self.k
                    out_row = out_c[i]
                    idx_row = idx_c[i]
                    
                    for j in range(ow):
                        jj = j * self.k
                        max_val = x_c[ii][jj]
                        max_pos = (ii, jj)
                        
                        for ki in range(self.k):
                            x_row = x_c[ii + ki]
                            for kj in range(self.k):
                                val = x_row[jj + kj]
                                if val > max_val:
                                    max_val = val
                                    max_pos = (ii + ki, jj + kj)
                        
                        out_row[j] = max_val
                        idx_row[j] = max_pos
        return out

    def backward(self, dout):
        b, c = len(dout), len(dout[0])
        oh, ow = len(dout[0][0]), len(dout[0][0][0])
        h, w = oh * self.k, ow * self.k
        
        dx = [[[[0.0]*w for _ in range(h)] for _ in range(c)] for _ in range(b)]
        
        for bi in range(b):
            dout_batch = dout[bi]
            dx_batch = dx[bi]
            idx_batch = self.max_idx[bi]
            
            for ci in range(c):
                dout_c = dout_batch[ci]
                dx_c = dx_batch[ci]
                idx_c = idx_batch[ci]
                
                for i in range(oh):
                    dout_row = dout_c[i]
                    idx_row = idx_c[i]
                    
                    for j in range(ow):
                        ii, jj = idx_row[j]
                        dx_c[ii][jj] = dout_row[j]
        return dx


class TinyDense:
    def __init__(self, in_f, out_f):
        self.in_f, self.out_f = in_f, out_f
        s = (2.0 / in_f) ** 0.5
        self.w = [[random.gauss(0, s) for _ in range(in_f)] for _ in range(out_f)]
        self.b = [0.0] * out_f
        self.x = None

    def forward(self, x):
        self.x = x
        b = len(x)
        out = [[self.b[j] for j in range(self.out_f)] for _ in range(b)]
        
        for bi in range(b):
            x_b = x[bi]
            out_b = out[bi]
            
            for j in range(self.out_f):
                w_j = self.w[j]
                s = 0.0
                for i in range(self.in_f):
                    s += x_b[i] * w_j[i]
                out_b[j] += s
        return out

    def backward(self, dout, lr):
        b = len(dout)
        norm = lr / b
        
        d_w = [[0.0 for _ in range(self.in_f)] for _ in range(self.out_f)]
        d_b = [0.0] * self.out_f
        dx = [[0.0 for _ in range(self.in_f)] for _ in range(b)]
        
        for j in range(self.out_f):
            gb = 0.0
            w_j = self.w[j]
            d_w_j = d_w[j]
            
            for bi in range(b):
                grad = dout[bi][j]
                if grad > 5.0:
                    grad = 5.0
                elif grad < -5.0:
                    grad = -5.0
                
                gb += grad
                x_b = self.x[bi]
                dx_b = dx[bi]
                
                for i in range(self.in_f):
                    d_w_j[i] += grad * x_b[i]
                    dx_b[i] += grad * w_j[i]
            
            self.b[j] -= norm * gb
            
            for i in range(self.in_f):
                w_j[i] -= norm * d_w_j[i]
        
        return dx


def relu_fwd(x):
    return [[[[max(0.0, x[b][c][i][j])
               for j in range(len(x[0][0][0]))]
              for i in range(len(x[0][0]))]
             for c in range(len(x[0]))]
            for b in range(len(x))]

def relu_bwd(dout, x_pre):
    return [[[[dout[b][c][i][j] if x_pre[b][c][i][j] > 0 else 0.0
               for j in range(len(dout[0][0][0]))]
              for i in range(len(dout[0][0]))]
             for c in range(len(dout[0]))]
            for b in range(len(dout))]

def flatten_fwd(x):
    return [[x[b][c][i][j]
             for c in range(len(x[0]))
             for i in range(len(x[0][0]))
             for j in range(len(x[0][0][0]))]
            for b in range(len(x))]

def unflatten(dx, c, h, w):
    out = [[[[0.0]*w for _ in range(h)] for _ in range(c)] for _ in range(len(dx))]
    for b in range(len(dx)):
        dx_b = dx[b]
        out_b = out[b]
        idx = 0
        for ci in range(c):
            out_c = out_b[ci]
            for i in range(h):
                out_row = out_c[i]
                for j in range(w):
                    out_row[j] = dx_b[idx]
                    idx += 1
    return out

def ce_loss_grad(preds, tgts):
    b = len(preds)
    loss = 0.0
    grads = [[0.0]*len(preds[0]) for _ in range(b)]
    
    for i in range(b):
        pred_i = preds[i]
        mx = max(pred_i)
        ex = [math.exp(p - mx) for p in pred_i]
        s = sum(ex)
        inv_s = 1.0 / s
        probs = [e * inv_s for e in ex]
        
        loss -= math.log(probs[tgts[i]] + 1e-8)
        
        grad_i = grads[i]
        for j in range(len(pred_i)):
            grad_i[j] = probs[j]
        grad_i[tgts[i]] -= 1.0
    
    return loss / b, grads

def softmax_fwd(x):
    out = []
    for b in x:
        mx = max(b)
        ex = [math.exp(v - mx) for v in b]
        s = sum(ex)
        inv_s = 1.0 / s
        out.append([e * inv_s for e in ex])
    return out

def prep_files(path, ratio=0.8):
    fps, lbls = [], []
    classes = sorted([d for d in os.listdir(path)
                      if os.path.isdir(os.path.join(path, d))])

    if not classes:
        print(f"ERROR: No class directories found in {path}")
        print(f"Please check the path is correct!")
        return [], [], [], [], 0

    for idx, cls in enumerate(classes):
        cpath = os.path.join(path, cls)
        for name in os.listdir(cpath):
            if not name.endswith(('.png','.jpg','.jpeg')): continue
            fps.append(os.path.join(cpath, name))
            lbls.append(idx)

    if not fps:
        print(f"ERROR: No images found in {path}")
        print(f"Checked {len(classes)} class folders but found no .png/.jpg/.jpeg files")
        return [], [], [], [], 0

    combined = list(zip(fps, lbls))
    random.shuffle(combined)
    fps, lbls = zip(*combined)
    fps, lbls = list(fps), list(lbls)

    split = int(len(fps) * ratio)
    print(f"Train: {split}, Test: {len(fps)-split}, Classes: {len(classes)}")
    return fps[:split], lbls[:split], fps[split:], lbls[split:], len(classes)

def load_batch(paths, labels, indices):
    imgs, lbls = [], []
    for i in indices:
        img = load_img(paths[i])
        if img is not None:
            imgs.append(img)
            lbls.append(labels[i])
    return imgs, lbls

def train(tr_p, tr_l, n_cls, epochs=2, bs=256, lr=0.08):
    # Pool BEFORE conv TO REDUCE MACs
    num_filters = 4
    
    pool0 = TinyPool2D(2)  # 32x32 -> 16x16 FIRST (0 MACs!)
    conv = TinyConv2D(3, num_filters, 3, 1, h=16, w=16)  # Conv on 16x16 (4x fewer MACs!)
    pool1 = TinyPool2D(2)  # 16x16 -> 8x8
    dense_input = num_filters * 8 * 8  # 256 features
    dense = TinyDense(dense_input, n_cls)

    # Calculate MACs
    conv_macs = 3 * num_filters * 9 * 16 * 16  # On 16x16!
    dense_macs = dense_input * n_cls
    total_macs = conv_macs + dense_macs
    params = (3 * num_filters * 9 + num_filters) + (dense_input * n_cls + n_cls)
    
    print(f"\nMOBILENET-INSPIRED CNN:")
    print(f"  Architecture: Pool(32→16) → Conv(16×16, 3→{num_filters}) → Pool(16→8) → Dense({dense_input}→{n_cls})")
    print(f"\nCOMPLEXITY:")
    print(f"  Conv MACs:  {conv_macs:,} (on 16×16, not 32×32!)")
    print(f"  Dense MACs: {dense_macs:,}")
    print(f"  ─────────────────────────")
    print(f"  TOTAL Params: {params:,}")
    print(f"  TOTAL MACs:   {total_macs:,}")
    print(f"  TOTAL FLOPs:  {total_macs*2:,}")
    print(f"\nTRAINING:")
    print(f"  Epochs: {epochs}, Batch: {bs}, LR: {lr}")
    print(f"  Expected: ~25-30 min/epoch\n")

    for ep in range(epochs):
        t0 = time.time()
        loss_sum = corr = tot = 0

        idx = list(range(len(tr_p)))
        random.shuffle(idx)
        
        num_batches = len(tr_p) // bs

        for i in range(0, len(tr_p), bs):
            bi = idx[i:i+bs]
            bx, blbl = load_batch(tr_p, tr_l, bi)
            if not bx: continue
            
            batch_num = i // bs
            if batch_num % 30 == 0:
                elapsed = time.time() - t0
                rate = batch_num / elapsed if elapsed > 0 else 0
                print(f"  Batch {batch_num}/{num_batches} | Elapsed: {elapsed/60:.1f}min )

            # Forward
            p0 = pool0.forward(bx)  # Pool FIRST
            c1 = conv.forward(p0)   # Conv on 16x16
            c1r = relu_fwd(c1)
            p1 = pool1.forward(c1r)
            fl = flatten_fwd(p1)
            out = dense.forward(fl)

            # Loss
            loss, dout = ce_loss_grad(out, blbl)
            loss_sum += loss * len(bx)
            tot += len(bx)

            # Backward
            dx_flat = dense.backward(dout, lr)
            dx_unflat = unflatten(dx_flat, num_filters, 8, 8)
            dx_pool1 = pool1.backward(dx_unflat)
            dx_relu = relu_bwd(dx_pool1, c1)
            conv.backward(dx_relu, lr)

            # Accuracy
            preds = softmax_fwd(out)
            for j, p in enumerate(preds):
                if p.index(max(p)) == blbl[j]: corr += 1

        print(f"\nEpoch {ep+1}/{epochs} | Loss: {loss_sum/tot:.4f} | "
              f"Acc: {100.0*corr/tot:.2f}% | Time: {(time.time()-t0)/60:.1f}min")

    return pool0, conv, pool1, dense

def evaluate(te_p, te_l, pool0, conv, pool1, dense, bs=256):
    corr = tot = 0
    for i in range(0, len(te_p), bs):
        bi = list(range(i, min(i+bs, len(te_p))))
        bx, blbl = load_batch(te_p, te_l, bi)
        if not bx: continue

        p0 = pool0.forward(bx)
        c1 = conv.forward(p0)
        c1r = relu_fwd(c1)
        p1 = pool1.forward(c1r)
        fl = flatten_fwd(p1)
        out = dense.forward(fl)
        preds = softmax_fwd(out)

        for j, p in enumerate(preds):
            if p.index(max(p)) == blbl[j]: corr += 1
        tot += len(bx)

    print(f"\nTest Acc: {100.0*corr/tot:.2f}%")

# ============================================================================
# MAIN
# ============================================================================

path = '/scratch/karthikl_NEW/aditig/ERA5_weekly/data_2/data_2/'

print("="*70)
print("ULTRA-LOW MAC/FLOP CNN")
print("="*70)

tr_p, tr_l, te_p, te_l, n_cls = prep_files(path)

t0 = time.time()
pool0, conv, pool1, dense = train(tr_p, tr_l, n_cls, epochs=2, bs=256, lr=0.08)
total_time = (time.time()-t0)/60

print(f"\n{'='*70}")
print(f"TRAINING COMPLETE: {total_time:.1f} min ({total_time/60:.2f} hours)")
print("="*70)

evaluate(te_p, te_l, pool0, conv, pool1, dense)

if total_time < 180:
    print(f"\n✓ SUCCESS: Completed in under 3 hours!")
    print(f"  Time used: {total_time:.1f} min")
    print(f"  Margin: {180-total_time:.1f} min remaining")
else:
    print(f"\n⚠ Exceeded 3 hour limit: {total_time:.1f} min")

ULTRA-LOW MAC/FLOP CNN
Train: 38543, Test: 9636, Classes: 97

ULTRA-LOW MAC/FLOP CNN:
  Architecture: Pool(32→16) → Conv(16×16, 3→4) → Pool(16→8) → Dense(256→97)

COMPLEXITY:
  Conv MACs:  27,648 (on 16×16, not 32×32!)
  Dense MACs: 24,832
  ─────────────────────────
  TOTAL Params: 25,041
  TOTAL MACs:   52,480
  TOTAL FLOPs:  104,960

TRAINING:
  Epochs: 2, Batch: 256, LR: 0.08
  Expected: ~25-30 min/epoch

  Batch 150/150 | Elapsed: 17.7min | ETA: 0.0min
Epoch 1/2 | Loss: 4.5465 | Acc: 2.12% | Time: 17.7min
  Batch 150/150 | Elapsed: 11.9min | ETA: 0.0min
Epoch 2/2 | Loss: 4.3818 | Acc: 4.50% | Time: 11.9min

TRAINING COMPLETE: 29.7 min (0.49 hours)

Test Acc: 5.48%

✓ SUCCESS: Completed in under 3 hours!
  Time used: 29.7 min
  Margin: 150.3 min remaining
